# Étape 4 — Premières générations COV sur T4

**Un prompt dev, seed 0, luminance cible 0,7.** SD1.5, 512 × 512, DDIM 50, eta=0, CFG=7.5.
Le code commun reste dans `baseline.py`, `rewards.py` et le nouveau `guidance_cov.py`.

**Deux questions séparées :** les équations sont-elles correctement implémentées ?
La récompense finale s'améliore-t-elle sur cette image ? Une baisse de récompense n'est pas
un échec algorithmique. Les deux résultats sont conservés dans des champs distincts.

1. Ouvrir ce notebook dans Colab, choisir **GPU T4** et importer `diffusion-reward-guidance.zip` dans la première cellule.
2. Exécuter les deux premières cellules de code. **Seulement si des paquets ont été installés**, redémarrer
   la session et reprendre depuis la première cellule. Sinon, poursuivre directement.
3. Exécuter dans l'ordre : tests, BASE, identités, calibration, premier COV, deux autres intensités.
4. Télécharger le ZIP de résultats à la fin. La cellule d'export peut être exécutée après un échec.

Les tests CPU sont décrits dans `validation/`. **Aucun résultat COV T4 n'est prérempli.**
Le premier essai conserve U-Net fp16 + VAE fp32 sur GPU, sans checkpointing ni offload automatique.
Un OOM sera enregistré et devra être analysé avant de changer ce choix.

In [ ]:
from pathlib import Path
import os
import sys
import zipfile

candidates = [Path.cwd(), Path.cwd().parent, Path('/content/diffusion-reward-guidance')]
PROJECT_DIR = next((p for p in candidates if (p / 'guidance_cov.py').is_file()), None)
if PROJECT_DIR is None:
    from google.colab import files
    print('Importer diffusion-reward-guidance.zip.')
    uploaded = files.upload()
    archive_name = next(name for name in uploaded if name.endswith('.zip'))
    destination = Path('/content').resolve()
    with zipfile.ZipFile(archive_name) as archive:
        for entry in archive.infolist():
            (destination / entry.filename).resolve().relative_to(destination)
        archive.extractall(destination)
    PROJECT_DIR = destination / 'diffusion-reward-guidance'
assert (PROJECT_DIR / 'guidance_cov.py').is_file(), 'Archive COV requise.'
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print('Projet :', PROJECT_DIR)

## 1. Environnement identique à BASE

COV luminance n'ajoute aucune bibliothèque. ImageReward n'est ni chargé ni exécuté.
Les sorties pip sont affichées et sauvegardées. Une session issue du notebook 02 avec les
versions conformes ne nécessite aucune réinstallation ; utiliser néanmoins ce notebook dans
un runtime dédié évite de garder ImageReward et les anciens gradients en VRAM.

In [ ]:
import importlib.metadata
import subprocess

assert (3, 10) <= sys.version_info[:2] <= (3, 13), 'Python 3.10 à 3.13 requis.'
expected = dict(line.strip().split('==') for line in
                (PROJECT_DIR / 'requirements.txt').read_text(encoding='utf-8').splitlines()
                if line.strip() and not line.lstrip().startswith('#'))
torch_expected = {'torch': '2.8.0+cu126', 'torchvision': '0.23.0+cu126', 'torchaudio': '2.8.0+cu126'}
expected.update(torch_expected)
installed = {}
for package in expected:
    try:
        installed[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed[package] = None
needs_install = any(installed[p] != v for p, v in expected.items())
RESTART_REQUIRED = globals().get('RESTART_REQUIRED', False) or needs_install

def run_visible(command, log_name):
    log_dir = PROJECT_DIR / 'installation_logs'
    log_dir.mkdir(exist_ok=True)
    with (log_dir / log_name).open('w', encoding='utf-8') as log:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
            return_code = process.wait()
    if return_code:
        raise RuntimeError(f'Échec {return_code} : voir les lignes ci-dessus et {log_dir / log_name}.')

if needs_install:
    if any(installed[p] != v for p, v in torch_expected.items()):
        run_visible([sys.executable, '-m', 'pip', 'install', '--progress-bar', 'off', '--only-binary=:all:',
                     'torch==2.8.0', 'torchvision==0.23.0', 'torchaudio==2.8.0',
                     '--index-url', 'https://download.pytorch.org/whl/cu126'], 'cov-pytorch.log')
    run_visible([sys.executable, '-m', 'pip', 'install', '--progress-bar', 'off', '--only-binary=:all:',
                 '-r', 'requirements.txt'], 'cov-dependencies.log')
print('STOP : redémarrer la session puis reprendre au début.' if RESTART_REQUIRED
      else 'Versions conformes : poursuivre, sans redémarrage.')

In [ ]:
assert not RESTART_REQUIRED, 'Redémarrer réellement la session avant les imports.'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import csv
import gc
import hashlib
import inspect
import json
import math
import platform
import shutil
import time
from datetime import datetime, timezone
import torch
from PIL import Image, ImageDraw
from IPython.display import display
from baseline import (MODEL_ID, MODEL_REVISION, configure_torch, load_pipeline,
                      encode_prompt_cfg, make_initial_noise, sample_ddim, measure_call)
from rewards import decode_latents_differentiable, luminance_map, luminance_reward
from guidance_cov import isotropic_guidance, cov_noise_prediction, sample_cov

configure_torch()
assert torch.cuda.is_available(), 'Sélectionner un GPU T4 dans Colab.'
print(torch.cuda.get_device_name(0), torch.__version__, torch.version.cuda)
RUN_DIR = PROJECT_DIR / 'runs' / datetime.now(timezone.utc).strftime('cov_%Y%m%d_%H%M%S_%f_UTC')
RUN_DIR.mkdir(parents=True, exist_ok=False)
results = []
algorithm_checks = {'gpu_status': 'pending', 'trajectories': {}}

def save_json(name, value):
    (RUN_DIR / name).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')

run_visible([sys.executable, '-m', 'unittest', '-v', 'test_baseline', 'test_rewards', 'test_cov'], 'cov-tests.log')
shutil.copy2(PROJECT_DIR / 'installation_logs' / 'cov-tests.log', RUN_DIR / 'cpu_tests.txt')
print('Dossier résultats :', RUN_DIR)

## 2. Modèle, mêmes entrées et précision explicite

Le VAE est converti en fp32 **pour tous les décodages**, BASE comprise. Cela conserve les poids
de la variante fp16 déjà validée ; le cast n'invente pas des poids de précision supérieure.
Le U-Net et les états DDIM restent fp16. Les gradients et la soustraction de bruit sont fp32,
puis le bruit corrigé est converti en fp16 : ses normes avant/après conversion sont enregistrées.

Les trois fichiers de prompts servent uniquement à vérifier le split ; une seule image dev est utilisée.
La chauffe de deux étapes est exclue du chronométrage. Les temps suivants incluent sampler, diagnostics,
traces CPU et décodage final, mais pas écriture disque, chargement des poids ou encodage du texte.

In [ ]:
SEED, NUM_STEPS, CFG, TARGET, TAU2 = 0, 50, 7.5, 0.7, 1.0
dev = (PROJECT_DIR / 'data/prompts_50_dev.txt').read_text(encoding='utf-8-sig').splitlines()
heldout = (PROJECT_DIR / 'data/prompts_50_heldout.txt').read_text(encoding='utf-8-sig').splitlines()
all_prompts = (PROJECT_DIR / 'data/generation_prompts.txt').read_text(encoding='utf-8-sig').splitlines()
assert not set(dev) & set(heldout)
PROMPT = dev[0]
assert PROMPT in all_prompts
pipe = load_pipeline()
pipe.vae.to(dtype=torch.float32).eval().requires_grad_(False)
assert not pipe.vae.is_gradient_checkpointing
prompt_embeds, negative_prompt_embeds = encode_prompt_cfg(pipe, PROMPT)
initial_noise = make_initial_noise(pipe, SEED)
assert initial_noise.shape == (1, 4, 64, 64)
assert torch.equal(initial_noise, make_initial_noise(pipe, SEED))

def reward_fn(images):
    return luminance_reward(images, target=TARGET)

torch.save({'initial_noise': initial_noise.cpu(), 'prompt_embeds': prompt_embeds.cpu(),
            'negative_prompt_embeds': negative_prompt_embeds.cpu()}, RUN_DIR / 'inputs.pt')
with torch.no_grad():
    warm_latents, _ = sample_ddim(pipe, initial_noise, prompt_embeds, negative_prompt_embeds, num_steps=2)
    warm_image = decode_latents_differentiable(pipe.vae, warm_latents.float())
del warm_latents, warm_image
gc.collect()
torch.cuda.empty_cache()
pipe.scheduler.set_timesteps(NUM_STEPS, device=initial_noise.device)
source_files = ['baseline.py', 'rewards.py', 'guidance_cov.py', 'test_baseline.py',
                'test_rewards.py', 'test_cov.py', 'requirements.txt', 'notebooks/03_cov_t4.ipynb']
snapshot = RUN_DIR / 'source'
for name in source_files:
    (snapshot / name).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(PROJECT_DIR / name, snapshot / name)
environment = {
    'model_id': MODEL_ID, 'model_revision': MODEL_REVISION, 'prompt': PROMPT, 'seed': SEED,
    'resolution': [512, 512], 'num_steps': NUM_STEPS, 'eta': 0, 'cfg': CFG, 'target': TARGET, 'tau2': TAU2,
    'unet_dtype': str(pipe.unet.dtype), 'vae_dtype': str(pipe.vae.dtype), 'state_dtype': str(initial_noise.dtype),
    'correction_dtype': 'torch.float32; cast to state dtype before DDIM',
    'gradient_checkpointing': False, 'offload': False, 'reward': 'luminance',
    'scheduler': dict(pipe.scheduler.config), 'timesteps': pipe.scheduler.timesteps.cpu().tolist(),
    'gpu': torch.cuda.get_device_name(0), 'vram_mib': torch.cuda.get_device_properties(0).total_memory / 2**20,
    'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda,
    'deterministic_algorithms': torch.are_deterministic_algorithms_enabled(),
    'versions': {package: importlib.metadata.version(package) for package in expected},
    'source_sha256': {name: hashlib.sha256((PROJECT_DIR / name).read_bytes()).hexdigest() for name in source_files},
    'prompt_sha256': {name: hashlib.sha256((PROJECT_DIR / 'data' / name).read_bytes()).hexdigest()
                      for name in ['generation_prompts.txt', 'prompts_50_dev.txt', 'prompts_50_heldout.txt']},
    'cost_scope': 'sampler + diagnostics + CPU trace + final fp32 VAE decode; excludes disk/model load/text encoding/warmup',
}
save_json('environment.json', environment)
(RUN_DIR / 'pip_freeze.txt').write_text(subprocess.check_output(
    [sys.executable, '-m', 'pip', 'freeze'], text=True), encoding='utf-8')
print(PROMPT)
print(inspect.getsource(isotropic_guidance))

## 3. BASE et fonction de mesure commune

La fonction ci-dessous appelle BASE ou COV et les mêmes décodage et score finaux.
`delta_reward_vs_base` est une observation, **jamais une assertion**.
Tous les pas COV restent consultables en JSON et CSV. Les valeurs de récompense intermédiaires
portent sur $D(\mu_t)$ ; elles ne sont pas les récompenses finales d'une trajectoire.

In [ ]:
def run_generation(label, beta=None, selected_reward=reward_fn):
    step_reports = []
    if (RUN_DIR / f'{label}_steps.json').exists():
        raise FileExistsError(f'{label} déjà enregistré. Créer un nouveau RUN_DIR pour répéter sans écraser.')

    def generate_and_decode():
        if beta is None:
            final_latents, trace = sample_ddim(
                pipe, initial_noise, prompt_embeds, negative_prompt_embeds,
                num_steps=NUM_STEPS, guidance_scale=CFG, return_trace=True)
        else:
            final_latents, trace, _ = sample_cov(
                pipe, initial_noise, prompt_embeds, negative_prompt_embeds, selected_reward,
                beta=beta, tau2=TAU2, num_steps=NUM_STEPS, guidance_scale=CFG,
                return_trace=True, diagnostics=step_reports)
        with torch.no_grad():
            image = decode_latents_differentiable(pipe.vae, final_latents.float()).cpu()
        return final_latents.cpu(), trace, image

    started = time.perf_counter()
    try:
        (latents, trace, image), cost = measure_call(generate_and_decode)
        if not torch.isfinite(latents).all() or not torch.isfinite(image).all():
            raise FloatingPointError('Latents ou image finale non finis.')
        mean_luminance = luminance_map(image).mean().item()
        reward = reward_fn(image).item()
    except Exception as error:
        save_json(f'{label}_failure.json', {'status': 'failed', 'type': type(error).__name__,
                  'message': str(error), 'elapsed_seconds': time.perf_counter() - started})
        save_json(f'{label}_steps.json', step_reports)
        if isinstance(error, torch.cuda.OutOfMemoryError):
            (RUN_DIR / f'{label}_OOM.txt').write_text(torch.cuda.memory_summary(), encoding='utf-8')
        algorithm_checks['gpu_status'] = 'failed'
        save_json('algorithm_checks.json', algorithm_checks)
        raise

    base_reward = results[0]['reward'] if results else reward
    row = dict(label=label, beta=beta, tau2=TAU2, reward=reward,
               delta_reward_vs_base=reward-base_reward, mean_luminance=mean_luminance,
               signed_error=mean_luminance-TARGET, absolute_error=abs(mean_luminance-TARGET),
               latents_finite=bool(torch.isfinite(latents).all()),
               image_finite=bool(torch.isfinite(image).all()),
               strong_correction_steps=sum(s.get('strong_correction', False) for s in step_reports),
               max_relative_correction=max((s.get('relative_noise_correction', 0) for s in step_reports), default=0),
               **cost)
    tensor_data = {'latents': latents, 'trace': trace, 'image': image}
    torch.save(tensor_data, RUN_DIR / f'{label}_tensors.pt')
    pixels = (image[0].permute(1, 2, 0).numpy() * 255).round().clip(0, 255).astype('uint8')
    Image.fromarray(pixels).save(RUN_DIR / f'{label}.png')
    save_json(f'{label}_steps.json', step_reports)
    if step_reports:
        with (RUN_DIR / f'{label}_steps.csv').open('w', newline='', encoding='utf-8') as stream:
            writer = csv.DictWriter(stream, fieldnames=list(step_reports[0]))
            writer.writeheader()
            writer.writerows(step_reports)
    results.append(row)
    algorithm_checks['trajectories'][label] = {
        'completed_and_finite': row['latents_finite'] and row['image_finite'],
        'steps_valid': beta is None or (len(step_reports) == NUM_STEPS and all(
            s['finite'] and s['unet_graph_absent'] and s['state_detached'] for s in step_reports)),
        'reward_connected': beta is None or beta == 0 or all(s['reward_connected'] for s in step_reports),
    }
    save_json('algorithm_checks.json', algorithm_checks)
    save_json('results.json', results)
    with (RUN_DIR / 'results.csv').open('w', newline='', encoding='utf-8') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(row))
        writer.writeheader()
        writer.writerows(results)
    print(json.dumps(row, indent=2))
    return tensor_data, step_reports

base, _ = run_generation('BASE')
display(Image.open(RUN_DIR / 'BASE.png'))

## 4. Identités obligatoires sur les 50 états

À $\beta=0$, le gradient n'est pas évalué : le champ de norme vaut `null`, pas « gradient nul mesuré ».
Pour la récompense constante reliée au graphe, le gradient est calculé et doit être nul.
Ces comparaisons exigent **l'égalité exacte**, puisque le pas DDIM garde les opérations de BASE.
Elles ne dépendent pas du score de l'image.

In [ ]:
zero, zero_steps = run_generation('COV_beta0', beta=0.0)
constant, constant_steps = run_generation(
    'COV_constant', beta=1.0,
    selected_reward=lambda images: 1 + 0 * images.mean(dim=(1, 2, 3)))
for label, candidate in [('beta0', zero), ('constant', constant)]:
    errors = [(a.float()-b.float()).abs().max().item() for a, b in zip(base['trace'], candidate['trace'])]
    exact = (len(candidate['trace']) == len(base['trace']) == NUM_STEPS
             and all(torch.equal(a, b) for a, b in zip(base['trace'], candidate['trace'])))
    algorithm_checks[label] = {'all_states_exact': exact, 'per_step_max_abs': errors,
                               'final_latents_exact': torch.equal(base['latents'], candidate['latents']),
                               'images_exact': torch.equal(base['image'], candidate['image'])}
algorithm_checks['constant_gradient_zero'] = all(row['gradient_l2'] == 0 for row in constant_steps)
identities_passed = all(algorithm_checks[label][key] for label in ('beta0', 'constant')
                       for key in ('all_states_exact', 'final_latents_exact', 'images_exact'))
algorithm_checks['identities_passed'] = identities_passed and algorithm_checks['constant_gradient_zero']
if not algorithm_checks['identities_passed']:
    algorithm_checks['gpu_status'] = 'failed'
save_json('algorithm_checks.json', algorithm_checks)
assert algorithm_checks['identities_passed'], 'Identité BASE non validée.'
del zero, zero_steps, constant, constant_steps
print('Identités exactes : beta=0 et récompense constante.')

## 5. Calibrer une petite grille avant toute génération guidée

Sur trois états **de BASE** (premier, milieu, dernier), mesurer le gradient et
$q_t=\|\sigma_t\kappa_t v_t\|_2/\|\epsilon_{CFG}\|_2$ à $\beta=1$.
On fixe ensuite trois intensités pour que le maximum de ces trois ratios soit **1 %, 3 %, 10 %**.
Ces fractions sont des choix d'exploration explicites, pas des intensités optimales ou garanties stables.
Le coût des sondes est enregistré séparément. Aucun score d'une image COV ne sert à choisir cette grille.

Le ratio est pris **avant** quantification fp16. Pendant les trajectoires, enregistrer aussi la correction
réellement appliquée, les pixels saturés et les pas dépassant 10 % (seuil de diagnostic, aucun clipping).
Trois sondes ne bornent pas toutes les corrections de la trajectoire guidée.

In [ ]:
PROBE_INDICES = [0, NUM_STEPS // 2 - 1, NUM_STEPS - 1]
def probe_base_gradients():
    probes = []
    pipe.scheduler.set_timesteps(NUM_STEPS, device=initial_noise.device)
    with torch.no_grad():
        embeddings = torch.cat([negative_prompt_embeds, prompt_embeds])
        for index in PROBE_INDICES:
            timestep = pipe.scheduler.timesteps[index]
            # trace[i] est l'état APRES le pas i ; récupérer ici l'état AVANT le pas.
            z_t = (initial_noise * pipe.scheduler.init_noise_sigma if index == 0
                   else base['trace'][index-1].to(initial_noise.device))
            latent_input = pipe.scheduler.scale_model_input(torch.cat([z_t, z_t]), timestep)
            noise = pipe.unet(latent_input, timestep, encoder_hidden_states=embeddings, return_dict=False)[0]
            noise_u, noise_c = noise.chunk(2)
            noise_cfg = noise_u + CFG * (noise_c-noise_u)
            _, report = cov_noise_prediction(pipe.vae, z_t, noise_cfg, timestep,
                                             pipe.scheduler, reward_fn, beta=1.0, tau2=TAU2)
            report['step_index'] = index
            probes.append(report)
    return probes

try:
    probes, probe_cost = measure_call(probe_base_gradients)
except Exception as error:
    save_json('calibration_failure.json', {'type': type(error).__name__, 'message': str(error)})
    if isinstance(error, torch.cuda.OutOfMemoryError):
        (RUN_DIR / 'calibration_OOM.txt').write_text(torch.cuda.memory_summary(), encoding='utf-8')
    algorithm_checks['gpu_status'] = 'failed'
    save_json('algorithm_checks.json', algorithm_checks)
    raise
save_json('calibration.json', {'probes': probes, 'cost': probe_cost, 'status': 'measured_before_grid'})
max_ratio = max(row['relative_noise_correction_fp32'] for row in probes)
if not math.isfinite(max_ratio) or max_ratio <= 0:
    save_json('calibration.json', {'probes': probes, 'cost': probe_cost, 'status': 'no_usable_probe_gradient'})
    raise RuntimeError('Grille non définie : examiner les sondes sauvegardées. Un gradient nul ne prouve pas une erreur COV.')
relative_targets = [0.01, 0.03, 0.10]
POSITIVE_BETAS = [fraction/max_ratio for fraction in relative_targets]
save_json('calibration.json', {'probes': probes, 'cost': probe_cost, 'relative_targets': relative_targets,
                              'positive_betas': POSITIVE_BETAS, 'rule': 'beta = target / max(q_t on three BASE states)'})
print(json.dumps({'probes': probes, 'positive_betas': POSITIVE_BETAS, 'cost': probe_cost}, indent=2))

## 6. Premier COV, une image

Chaîne de calcul visible dans `cov_noise_prediction` :

$$\epsilon_{CFG}\to\mu_t\to R(D(\mu_t))\to v_t\to
g_t=\beta\frac{\alpha_t\tau^2}{\sigma_t^2+\alpha_t^2\tau^2}v_t
\to\epsilon_{CFG}-\sigma_tg_t\to DDIM.$$

La récompense utilise la prédiction propre **brute** en fp32. Le pas DDIM applique ensuite ses
conventions, dont l'éventuel clipping ; il emploie le bruit corrigé dans ses deux occurrences.
Pour SD1.5, `clip_sample=False`. Les états sont détachés à chaque pas et les journaux contiennent
seulement des nombres : aucun graphe de gradient n'est gardé entre les pas.

In [ ]:
first_cov, first_steps = run_generation('COV_1', beta=POSITIVE_BETAS[0])
display(Image.open(RUN_DIR / 'COV_1.png'))
print('Gain de récompense observé :', results[-1]['delta_reward_vs_base'])
print('Pas avec correction > 10 % :', results[-1]['strong_correction_steps'])
print('VRAM allouée après chaque pas (MiB) :', [round(s['allocated_after_step_mib'], 1) for s in first_steps])
assert all(s['finite'] and s['unet_graph_absent'] and s['state_detached'] for s in first_steps)
assert all(s['reward_connected'] for s in first_steps), 'La luminance devrait rester reliée au latent.'
del first_cov, first_steps

## 7. Les deux autres intensités et contrôles finaux

La grille a déjà été figée. Toutes les générations sont conservées, même si elles sont moins bonnes.
Une norme de gradient nulle à un état stationnaire ou une baisse de score final ne fait pas échouer COV.
Une erreur numérique, un graphe U-Net ou un paramètre qui accumule un gradient est un échec à examiner.
La mémoire allouée après les étapes permet d'inspecter une éventuelle accumulation ; la mémoire réservée
par le cache CUDA n'est pas une preuve de fuite. Le pic réel de la trajectoire est mesuré séparément.

In [ ]:
for index, beta in enumerate(POSITIVE_BETAS[1:], start=2):
    generated, steps = run_generation(f'COV_{index}', beta=beta)
    assert len(steps) == NUM_STEPS
    assert all(s['finite'] and s['unet_graph_absent'] and s['state_detached'] and s['reward_connected'] for s in steps)
    del generated, steps

algorithm_checks['models_frozen'] = all(not p.requires_grad for model in (pipe.unet, pipe.vae) for p in model.parameters())
algorithm_checks['parameter_grads_none'] = all(p.grad is None for model in (pipe.unet, pipe.vae) for p in model.parameters())
algorithm_checks['all_images_finite'] = all(row['image_finite'] and row['latents_finite'] for row in results)
required_labels = ['BASE', 'COV_beta0', 'COV_constant', 'COV_1', 'COV_2', 'COV_3']
algorithm_checks['all_trajectories_valid'] = all(
    label in algorithm_checks['trajectories'] and all(algorithm_checks['trajectories'][label].values())
    for label in required_labels)
passed = (algorithm_checks.get('identities_passed', False) and algorithm_checks['models_frozen']
          and algorithm_checks['parameter_grads_none'] and algorithm_checks['all_images_finite']
          and algorithm_checks['all_trajectories_valid'])
algorithm_checks['gpu_status'] = 'passed_for_this_prompt_seed_and_grid' if passed else 'failed'
algorithm_checks['scope'] = 'identities, finite gradients, detached states, complete GPU trajectories; reward gain is separate'
save_json('algorithm_checks.json', algorithm_checks)
assert passed, 'Validation algorithmique incomplète ou échouée : lire algorithm_checks.json.'
print('Validation algorithmique GPU réussie pour ce pilote. Les performances restent les observations ci-dessous.')
for row in results:
    print(f"{row['label']:14s}  R={row['reward']:+.6f}  delta={row['delta_reward_vs_base']:+.6f}  "
          f"luminance={row['mean_luminance']:.4f}  {row['seconds']:.2f}s  "
          f"pic={row['peak_allocated_mib']/1024:.2f} GiB")

## 8. Comparaisons visuelles et interprétation

Chaque panneau est sauvegardé en PNG à la résolution originale, BASE à gauche et COV à droite.
Les métriques proviennent des images flottantes fp32 avant leur conversion PNG.

Si le score baisse : regarder l'échelle des corrections, les normes par timestep, la saturation,
et la différence entre la récompense de la prédiction propre et celle de l'image finale. COV emploie
une approximation locale du Jacobien et ne garantit pas d'amélioration globale. Un score qui monte
ne prouve pas non plus la qualité du contenu ni un échantillonnage fidèle à la distribution cible.
Un prompt et une seed constituent un pilote, pas une conclusion comparative générale.

In [ ]:
for row in [r for r in results if r['label'] in ('COV_1', 'COV_2', 'COV_3')]:
    panel = Image.new('RGB', (1024, 558), 'white')
    panel.paste(Image.open(RUN_DIR / 'BASE.png'), (0, 46))
    panel.paste(Image.open(RUN_DIR / f"{row['label']}.png"), (512, 46))
    draw = ImageDraw.Draw(panel)
    draw.text((10, 8), f"BASE | R={results[0]['reward']:.6f}", fill='black')
    draw.text((522, 8), f"{row['label']} | beta={row['beta']:.6g} | R={row['reward']:.6f}", fill='black')
    draw.text((522, 26), f"delta R={row['delta_reward_vs_base']:+.6f}", fill='black')
    panel.save(RUN_DIR / f"compare_{row['label']}.png")
    display(panel)

## 9. Export — également exécutable après un échec

Conserver ensemble configuration, sources, entrées, résultats, états intermédiaires et coûts.
L'archive ne contient aucun token. Elle inclut seulement les fichiers de ce run ; aucun ancien résultat
n'est écrasé. Envoyer cette archive pour examiner la validation et les images avant d'étendre le protocole.

In [ ]:
save_json('algorithm_checks.json', algorithm_checks)
save_json('results.json', results)
archive_path = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name)
print('Résultats :', archive_path)
try:
    from google.colab import files
except ImportError:
    print('Northflank/local : récupérer cette archive depuis le système de fichiers.')
else:
    files.download(archive_path)